# Session 10: Sanger Sequencing Analysis

**Module 3: Programming for Biological Data**  
**Date:** February 4, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Understand **Sanger sequencing** and chromatograms
2. Interpret **Phred quality scores**
3. Use **sangeranalyseR** to process AB1 files
4. Apply **quality trimming** to sequences
5. Export clean **FASTA** sequences

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL (Bioconductor)
# ============================================
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

# Install sangeranalyseR (may take 3-5 minutes)
if (!requireNamespace("sangeranalyseR", quietly = TRUE))
  BiocManager::install("sangeranalyseR", update = FALSE, ask = FALSE)

library(sangeranalyseR)

cat("✅ sangeranalyseR loaded successfully!")

---

# Part 1: The Anatomy of a Chromatogram

## 40 minutes

---

## 1.1 How Sanger Sequencing Works

**Sanger sequencing** (chain termination method):

1. DNA template + primer + DNA polymerase
2. Normal dNTPs + fluorescently labeled **ddNTPs** (terminators)
3. Random termination creates fragments of all lengths
4. **Capillary electrophoresis** separates by size
5. **Laser detection** reads fluorescent labels

Result: **Chromatogram** (trace file)

## 1.2 The AB1 File Format

**AB1** = Applied Biosystems proprietary format

Contains:
- Raw signal data (4 channels: A, T, G, C)
- Base calls
- Quality scores
- Metadata (sample name, date, etc.)

## 1.3 Phred Quality Scores

**Phred score** = measure of base call accuracy

$$Q = -10 \times \log_{10}(P_{error})$$

| Phred Score | Error Probability | Accuracy |
|-------------|-------------------|----------|
| Q10 | 1 in 10 | 90% |
| Q20 | 1 in 100 | 99% |
| Q30 | 1 in 1,000 | 99.9% |
| Q40 | 1 in 10,000 | 99.99% |

In [ ]:
# Calculate error probability from Phred score
phred_to_prob <- function(Q) {
  10^(-Q/10)
}

# Examples
cat("Q10 error rate:", phred_to_prob(10), "(1 in 10)\n")
cat("Q20 error rate:", phred_to_prob(20), "(1 in 100)\n")
cat("Q30 error rate:", phred_to_prob(30), "(1 in 1000)\n")

## 1.4 Reading a Chromatogram

**Key features to look for:**

| Feature | Good | Bad |
|---------|------|-----|
| Peaks | Sharp, well-separated | Broad, overlapping |
| Baseline | Flat | Noisy |
| Signal | Strong | Weak |
| Spacing | Even | Variable |

**Common problems:**
- Dye blobs (early contamination)
- Signal drop-off (late in read)
- Mixed peaks (heterozygotes or contamination)

## 1.5 Quality Issues in Sanger Data

**5' end issues:**
- Primer region (known sequence)
- Dye blob artifacts

**3' end issues:**
- Signal degradation
- Increased noise
- Lower quality scores

**Solution:** Quality trimming!

---

# Part 2: The sangeranalyseR Package

## 40 minutes

---

## 2.1 Why sangeranalyseR?

**sangeranalyseR** is a Bioconductor package for:
- Reading AB1 files
- Quality analysis
- Automated trimming
- Building contigs (Fwd + Rev)
- Generating reports

## 2.2 The Sanger Workflow

```
Raw AB1 file
     ↓
SangerRead() — Load & analyze
     ↓
Quality assessment
     ↓
Trim low-quality ends (Mott's algorithm)
     ↓
Export clean FASTA
```

## 2.3 Loading Real-World Data

We will analyze real **Influenza A (H1N1 and H3N2)** Sanger sequencing data from laboratory surveillance.

**Dataset:**
- **Genes:** Hemagglutinin (HA) and Neuraminidase (NA)
- **Format:** AB1 files (trace data)
- **Source:** Remote GitHub Repository

In [ ]:
# Define base URL for data
base_url <- "https://github.com/Koohoko/HKU_Space_Applied_Medical_Microbiology_Bioinformatics_tutorial_2024/raw/refs/heads/main/data/Sanger_sequencing/"

# Helper function to download and read AB1 from URL
read_ab1_from_url <- function(url) {
  if (!grepl("\\.ab1$", url, ignore.case = TRUE)) stop("Invalid .ab1 URL")
  temp_file <- tempfile(fileext = ".ab1")
  download.file(url, temp_file, mode="wb", quiet = TRUE)
  ab1_data <- SangerRead(readFileName = temp_file, readFeature = "Forward Read")
  unlink(temp_file)
  return(ab1_data)
}

cat("Data source defined:", base_url)

## 2.4 Reading a Single Read

We'll start by reading the forward read: `B2_S2_HA_FL_FW-.ab1`.

![](https://sangeranalyser.readthedocs.io/en/latest/_images/SangerRead_hierarchy.png)

In [ ]:
# Target file URL
fwd_url <- paste0(base_url, "B2_S2_HA_FL_FW-.ab1")

# Read from URL
sanger_read <- read_ab1_from_url(fwd_url)

# View summary
sanger_read

## 2.5 Visualizing Quality

Let's look at the quality scores across the read. We expect lower quality at the very beginning (dye blobs) and end (signal degradation).

In [ ]:
# Plot quality scores per base
qualityBasePlot(sanger_read)

In [ ]:
launchApp(sanger_read)

## 2.6 Trimming and Sequence Extraction

**sangeranalyseR** applies Mott's trimming algorithm by default when asking for the primary sequence.

Let's compare the raw vs. trimmed sequence lengths.

In [ ]:
# Get full raw sequence
raw_seq <- primarySeq(sanger_read, string = TRUE)

# Get trimming positions from QualityReport
start_pos <- sanger_read@QualityReport@trimmedStartPos
end_pos <- sanger_read@QualityReport@trimmedFinishPos

# Extract trimmed sequence
clean_seq <- substr(raw_seq, start_pos, end_pos)

cat("Raw Length:", nchar(raw_seq), "bp\n")
cat("Trimmed Length:", nchar(clean_seq), "bp\n")
cat("Bases removed:", nchar(raw_seq) - nchar(clean_seq), "bp")

In [ ]:
generateReport(sanger_read)

## 2.7 Building a Contig

To build a contig, we must have local files. We will download both Forward and Reverse reads to a temporary directory.

![](https://sangeranalyser.readthedocs.io/en/latest/_images/SangerContig_hierarchy.png)

In [ ]:
# Create a temporary directory for contig building
temp_contig_dir <- tempfile()
dir.create(temp_contig_dir)

# Download both files
fwd_local <- file.path(temp_contig_dir, "B2_S2_HA_FL_FW-.ab1")
rev_local <- file.path(temp_contig_dir, "B2_S2_HA_FL_RV-.ab1")

download.file(paste0(base_url, "B2_S2_HA_FL_FW-.ab1"), fwd_local, mode="wb", quiet=TRUE)
download.file(paste0(base_url, "B2_S2_HA_FL_RV-.ab1"), rev_local, mode="wb", quiet=TRUE)

# Create SangerContig object using local files
sanger_contig <- SangerContig(
    inputSource = "ABIF",
    ABIF_Directory = file.path(temp_contig_dir),
    processMethod = "REGEX",
    REGEX_SuffixForward = "_FW-.ab1$",
    REGEX_SuffixReverse = "_RV-.ab1$",
    contigName = "B2_S2_HA_FL"
)

# View contig summary
sanger_contig

## 2.8 Quality Report Generation

**sangeranalyseR** can generate an interactive HTML report for the contig, showing the alignment and quality of both reads.

*(Note: Report generation might take a minute)*

In [ ]:
# Generate quality report (saves to 'Report' folder)
# This creates an HTML file you can open in a browser
generateReport(sanger_contig, outputDir = "Sanger_Report")

In [ ]:
launchApp(sanger_contig)

---

# Key Takeaways

1. **Sanger sequencing** producing AB1 files is the gold standard for validating sequences.

2. **Real-world data** (Influenza) shows typical quality profiles: good in the middle, poor at ends.

3. **sangeranalyseR** makes it easy to:
   - Load files (`SangerRead`)
   - Check quality (`qualityBasePlot`)
   - Trim low-quality bases (`primarySeq` trimming logic)
   - Assemble forward/reverse reads (`SangerContig`)

---

## Now proceed to Tutorial 10! 🧬